# Proper Few-Shot WildFireVQA (Frozen VLM)

This notebook performs inference with a pretrained vision-language model and `NUM_SHOTS` multimodal demonstrations. It never trains on WildFireVQA: no optimizer, backward pass, adapter, or parameter update is created. Support examples and test queries are separated by image.


In [ ]:
!pip install -q -U 'transformers>=4.49.0' bitsandbytes accelerate kagglehub qwen-vl-utils sentencepiece


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
import kagglehub

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

print('Torch:', torch.__version__)
print('CUDA devices:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f'GPU {index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)')


In [ ]:
class CFG:
    KAGGLE_DATASET = 'caseypiere/wildfire-vqa'
    FLAME3_KAGGLE_DATASET = 'brycehopkins/flame-3-computer-vision-subset-sycan-marsh'
    DATA_ROOT = Path('/kaggle/input/datasets/caseypiere/wildfire-vqa')
    FLAME3_ROOT = None
    FLAME3_MARKER_DIR = 'Flame 3 Computer Vision Sets'
    ANNOTATION_FILE = None
    DATA_SOURCE = 'kagglehub'
    ALLOWED_SITE_PREFIX = 'Sycan'

    MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
    USE_4BIT = True
    NUM_SHOTS = 3  # Change to any k >= 1. No gradient updates are performed.
    TEST_FRAC = 0.10
    MAX_EVAL_SAMPLES = 200  # Set None for the complete held-out test split.
    MAX_NEW_TOKENS = 32
    IMAGE_SIZE = 224
    SEED = 42
    OUTPUT_DIR = Path('/kaggle/working/firevlm_frozen_few_shot_qwen25vl_3b')

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED)


In [ ]:
def load_json_any(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def flatten_checkpoint_json(obj: Dict[str, Any], source_name: str) -> List[Dict[str, Any]]:
    rows = []
    items = obj.get('items', {})
    if not isinstance(items, dict):
        return rows
    for ck, entry in items.items():
        if not isinstance(entry, dict):
            continue
        meta = entry.get('meta', {})
        if not isinstance(meta, dict):
            continue
        row = dict(meta)
        row['gt_answer'] = entry.get('human_answer', row.get('gt_answer', row.get('answer')))
        row['checkpoint_key'] = ck
        row['source_input'] = source_name
        rows.append(row)
    return rows

def normalize_options(x):
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
        except Exception:
            pass
        if '|' in s:
            return [v.strip() for v in s.split('|') if v.strip()]
        if ';' in s:
            return [v.strip() for v in s.split(';') if v.strip()]
        return [s]
    return []

def rows_from_json(path: Path) -> List[Dict[str, Any]]:
    obj = load_json_any(path)
    if isinstance(obj, dict) and obj.get('type') == 'checkpoint':
        return flatten_checkpoint_json(obj, str(path))
    if isinstance(obj, dict):
        for key in ['items', 'annotations', 'data', 'records', 'questions']:
            if isinstance(obj.get(key), list):
                return [r for r in obj[key] if isinstance(r, dict)]
    if isinstance(obj, list):
        return [r for r in obj if isinstance(r, dict)]
    return []

def discover_annotation_files(root: Path):
    exts = ['*.json', '*.jsonl', '*.csv', '*.parquet']
    files = []
    if root.exists():
        for ext in exts:
            files.extend(root.rglob(ext))
    priority = ('vqa', 'question', 'answer', 'annotation', 'response', 'train', 'val')
    return sorted(files, key=lambda p: (not any(w in p.name.lower() for w in priority), len(str(p))))

def load_rows_from_files():
    if CFG.ANNOTATION_FILE:
        files = [Path(CFG.ANNOTATION_FILE)]
    else:
        files = discover_annotation_files(CFG.DATA_ROOT)
    if not files:
        raise FileNotFoundError(f'No annotation files found under {CFG.DATA_ROOT}')

    all_rows = []
    for p in files:
        try:
            if p.suffix.lower() == '.json':
                rows = rows_from_json(p)
            elif p.suffix.lower() == '.jsonl':
                with open(p, 'r', encoding='utf-8') as f:
                    rows = [json.loads(line) for line in f if line.strip()]
            elif p.suffix.lower() == '.csv':
                rows = pd.read_csv(p).to_dict('records')
            elif p.suffix.lower() == '.parquet':
                rows = pd.read_parquet(p).to_dict('records')
            else:
                rows = []
            for r in rows:
                r.setdefault('source_input', str(p))
            all_rows.extend(rows)
        except Exception as e:
            print('[WARN] failed to load', p, repr(e))
    return all_rows

def load_rows_from_kagglehub():
    if kagglehub is None:
        raise ImportError('kagglehub is unavailable.')
    try:
        dataset_root = Path(kagglehub.dataset_download(CFG.KAGGLE_DATASET))
        CFG.DATA_ROOT = dataset_root
    except Exception as e:
        print('[WARN] kagglehub.dataset_download failed; using local path:', repr(e))
    return load_rows_from_files()

if CFG.DATA_SOURCE == 'kagglehub':
    raw_rows = load_rows_from_kagglehub()
else:
    raw_rows = load_rows_from_files()

print(f'Loaded {len(raw_rows):,} raw annotation rows.')


In [ ]:
def get_first(row, names, default=None):
    for n in names:
        if n in row and row[n] is not None:
            return row[n]
    return default

def normalize_answer_text(x):
    if isinstance(x, list):
        vals = [normalize_answer_text(v) for v in x]
        vals = [v for v in vals if v]
        return Counter(vals).most_common(1)[0][0] if vals else ''
    if isinstance(x, dict):
        for key in ['answer', 'label', 'text', 'gt_answer', 'human_answer']:
            if key in x:
                return normalize_answer_text(x[key])
        return json.dumps(x, sort_keys=True)
    return re.sub(r'\s+', ' ', str(x).strip()) if x is not None else ''

records = []
for row in raw_rows:
    question = get_first(row, ['question', 'query', 'prompt'], '')
    answer = get_first(row, ['gt_answer', 'answer', 'human_answer', 'label', 'target'], '')
    options = normalize_options(get_first(row, ['options', 'choices', 'candidate_answers'], []))
    rgb_path = get_first(row, ['rgb_path', 'image_path', 'image', 'img_path', 'rgb', 'file_name', 'filename'], None)
    thermal_path = get_first(row, ['thermal_path', 'thermal', 'ir_path', 'thermal_image', 'thermal_jpg'], None)
    temp_summary = get_first(row, ['temp_summary', 'temperature_summary', 'thermal_summary'], None)

    rec = {
        'image_id': str(get_first(row, ['image_id', 'frame_id', 'id'], '')),
        'question_id': str(get_first(row, ['question_id', 'qid'], '')),
        'category': str(get_first(row, ['category', 'question_category', 'group'], 'unknown')),
        'question': str(question).strip(),
        'answer': normalize_answer_text(answer),
        'options': options,
        'rgb_path': rgb_path,
        'thermal_path': thermal_path,
        'temp_summary': temp_summary if isinstance(temp_summary, dict) else None,
        'source_input': str(get_first(row, ['source_input'], '')),
    }
    if rec['question'] and rec['answer']:
        records.append(rec)

df = pd.DataFrame(records).drop_duplicates(
    subset=['image_id', 'question_id', 'question', 'answer', 'rgb_path', 'thermal_path']
).reset_index(drop=True)

# ---------------------------------------------------------------
# 1) Which FLAME-3 site does each annotation row come from?
# ---------------------------------------------------------------
MARKER = CFG.FLAME3_MARKER_DIR

def site_of(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return 'unknown'
    parts = str(p).replace('\\', '/').split('/')
    if MARKER in parts:
        i = parts.index(MARKER)
        if i + 1 < len(parts):
            return parts[i + 1]
    return 'unknown'

df['site'] = df['rgb_path'].apply(site_of)
print('Rows per site in annotations:')
print(df['site'].value_counts().to_string())

# Keep only sites present in the Kaggle subset (Sycan*).
df = df[df['site'].str.startswith(CFG.ALLOWED_SITE_PREFIX)].reset_index(drop=True)
print(f'\nRows after site filter ({CFG.ALLOWED_SITE_PREFIX}*): {len(df):,}')

# ---------------------------------------------------------------
# 2) Locate the folder on Kaggle that directly contains Fire/ and No Fire/
# ---------------------------------------------------------------
def discover_flame_base():
    roots = []
    if CFG.FLAME3_ROOT is not None:
        roots.append(Path(CFG.FLAME3_ROOT))
    if kagglehub is not None:
        try:
            roots.append(Path(kagglehub.dataset_download(CFG.FLAME3_KAGGLE_DATASET)))
        except Exception as e:
            print('[WARN] kagglehub flame3 download failed:', repr(e))
    if Path('/kaggle/input').exists():
        roots.append(Path('/kaggle/input'))
    for r in roots:
        if not r.exists():
            continue
        hits = sorted([p for p in r.rglob('Fire') if p.is_dir()], key=lambda p: len(p.parts))
        if hits:
            return hits[0].parent
    return None

FLAME_BASE = discover_flame_base()
assert FLAME_BASE is not None, 'Could not find a folder containing Fire/ in the FLAME-3 dataset.'
print('\nFLAME base folder:', FLAME_BASE)
print('Contents:', sorted(p.name for p in FLAME_BASE.iterdir()))

# ---------------------------------------------------------------
# 3) Strict structural path mapping (NO filename-only fallback)
#    annotation: .../<site>/Images/<Fire|No Fire>/<RGB|Thermal>/<sub>/<file>
#    kaggle:     <BASE>/<Fire|No Fire>/<RGB|Thermal>/<sub'>/<file>
# ---------------------------------------------------------------
SUBDIR_MAP = {
    ('RGB', 'Corrected FOV'): ('RGB', 'Corrected FOV'),
    ('RGB', 'Raw'):           ('RGB', 'Raw'),
    ('Thermal', 'JPG'):       ('Thermal', 'Raw JPG'),
    ('Thermal', 'Raw JPG'):   ('Thermal', 'Raw JPG'),
    ('Thermal', 'TIFF'):      ('Thermal', 'Celsius TIFF'),
    ('Thermal', 'Celsius TIFF'): ('Thermal', 'Celsius TIFF'),
}

def resolve_path(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    parts = str(value).replace('\\', '/').split('/')
    cls = next((c for c in ('Fire', 'No Fire') if c in parts), None)
    if cls is None:
        return None
    i = len(parts) - 1 - parts[::-1].index(cls)   # last occurrence of the class folder
    try:
        modality, sub, fname = parts[i + 1], parts[i + 2], parts[-1]
    except IndexError:
        return None
    m, s = SUBDIR_MAP.get((modality, sub), (modality, sub))
    cand = FLAME_BASE / cls / m / s / fname
    return str(cand) if cand.exists() else None

df['rgb_resolved'] = df['rgb_path'].apply(resolve_path)
df['thermal_resolved'] = df['thermal_path'].apply(resolve_path)

print(f'\nRGB resolved:     {df["rgb_resolved"].notna().mean():.1%}')
print(f'Thermal resolved: {df["thermal_resolved"].notna().mean():.1%}')

# Require BOTH modalities, and they must be different files
df = df[df['rgb_resolved'].notna() & df['thermal_resolved'].notna()]
df = df[df['rgb_resolved'] != df['thermal_resolved']].reset_index(drop=True)

# If two sites (e.g. Sycan_2A and Sycan_2D) map onto the same Kaggle file, the pairing is ambiguous -> drop
n_sites = df.groupby('rgb_resolved')['site'].nunique()
ambiguous = n_sites[n_sites > 1].index
if len(ambiguous):
    print(f'[WARN] {len(ambiguous):,} images are shared by multiple sites -> dropping those rows')
    df = df[~df['rgb_resolved'].isin(ambiguous)].reset_index(drop=True)

print(f'\nFinal rows with valid, distinct RGB + thermal: {len(df):,}')
print('Unique RGB images:', df['rgb_resolved'].nunique())
print('RGB == thermal fraction (must be 0):', (df['rgb_resolved'] == df['thermal_resolved']).mean())
assert len(df) > 0, 'No rows resolved. Check FLAME_BASE contents above.'
df[['site', 'question', 'answer', 'rgb_resolved', 'thermal_resolved']].head()

In [ ]:
# Support and test are split by image, preventing questions from one frame crossing splits.
unique_images = df['rgb_resolved'].unique().tolist()
rng = np.random.default_rng(CFG.SEED)
rng.shuffle(unique_images)
n_test = max(1, int(len(unique_images) * CFG.TEST_FRAC))
test_images = set(unique_images[:n_test])
support_df = df[~df['rgb_resolved'].isin(test_images)].reset_index(drop=True)
test_df = df[df['rgb_resolved'].isin(test_images)].reset_index(drop=True)
if CFG.MAX_EVAL_SAMPLES is not None and len(test_df) > CFG.MAX_EVAL_SAMPLES:
    test_df = test_df.sample(CFG.MAX_EVAL_SAMPLES, random_state=CFG.SEED).reset_index(drop=True)

assert set(support_df['rgb_resolved']).isdisjoint(set(test_df['rgb_resolved']))
assert set(support_df['thermal_resolved']).isdisjoint(set(test_df['thermal_resolved']))
assert CFG.NUM_SHOTS >= 1, 'This notebook is intentionally a proper few-shot experiment.'
print(f'Support pool: {len(support_df):,} rows')
print(f'Held-out test: {len(test_df):,} rows')
print(f'Few-shot demonstrations per query: {CFG.NUM_SHOTS}')

def load_image(path):
    with Image.open(path) as image:
        return image.convert('RGB').resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE), Image.BILINEAR)

# Pick support examples without reading test answers. Prefer the same question category,
# while ensuring that one image is not repeated within a prompt.
def select_demonstrations(query_row, query_index):
    candidates = support_df[support_df['category'] == query_row['category']]
    if len(candidates) < CFG.NUM_SHOTS:
        candidates = support_df
    candidates = candidates.drop_duplicates('rgb_resolved')
    if len(candidates) < CFG.NUM_SHOTS:
        raise ValueError('Not enough unique support images for NUM_SHOTS.')
    return candidates.sample(
        n=CFG.NUM_SHOTS,
        random_state=CFG.SEED + int(query_index),
    ).to_dict('records')


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator.')

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
) if CFG.USE_4BIT else None

processor = AutoProcessor.from_pretrained(
    CFG.MODEL,
    min_pixels=CFG.IMAGE_SIZE * CFG.IMAGE_SIZE,
    max_pixels=CFG.IMAGE_SIZE * CFG.IMAGE_SIZE,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG.MODEL,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print('Trainable parameters:', trainable)
assert trainable == 0, 'Few-shot baseline must not update model weights.'


In [ ]:
SYSTEM_PROMPT = (
    'You are a wildfire analyst. Answer the final question from its RGB and thermal aerial '
    'images. Use the demonstrations only as examples of the task. Return only a direct, concise answer.'
)

def image_pair_content(row, label):
    return [
        {'type': 'text', 'text': f'{label} RGB image:'},
        {'type': 'image', 'image': load_image(row['rgb_resolved'])},
        {'type': 'text', 'text': f'{label} thermal image:'},
        {'type': 'image', 'image': load_image(row['thermal_resolved'])},
    ]

def build_messages(query_row, demonstrations):
    messages = [{'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]}]
    for number, demo in enumerate(demonstrations, 1):
        content = image_pair_content(demo, f'Demonstration {number}')
        content.append({'type': 'text', 'text': f'Question: {demo["question"]}'})
        messages.append({'role': 'user', 'content': content})
        messages.append({'role': 'assistant', 'content': [{'type': 'text', 'text': demo['answer']}]})
    query_content = image_pair_content(query_row, 'Query')
    query_content.append({
        'type': 'text',
        'text': f'Question: {query_row["question"]}\nProvide a direct and concise answer.',
    })
    messages.append({'role': 'user', 'content': query_content})
    return messages

def prepare_inputs(messages):
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    return inputs.to('cuda:0')


In [ ]:
def simple_tokens(s: str) -> List[str]:
    return re.findall(r"\w+|[^\w\s]", str(s).lower())

def ngram_counts(tokens: List[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def sentence_bleu(reference: str, prediction: str, max_n: int = 4) -> float:
    ref = simple_tokens(reference)
    pred = simple_tokens(prediction)
    if not pred:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        pred_counts = ngram_counts(pred, n)
        ref_counts = ngram_counts(ref, n)
        if not pred_counts:
            precisions.append(1e-9)
            continue
        overlap = sum(min(c, ref_counts[g]) for g, c in pred_counts.items())
        precisions.append((overlap + 1) / (sum(pred_counts.values()) + 1))
    bp = 1.0 if len(pred) > len(ref) else math.exp(1 - len(ref) / max(len(pred), 1))
    return bp * math.exp(sum(math.log(p) for p in precisions) / max_n)

def lcs_len(a: List[str], b: List[str]) -> int:
    dp = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b, 1):
            cur = dp[j]
            if x == y:
                dp[j] = prev + 1
            else:
                dp[j] = max(dp[j], dp[j-1])
            prev = cur
    return dp[-1]

def rouge_l(reference: str, prediction: str) -> float:
    ref = simple_tokens(reference)
    pred = simple_tokens(prediction)
    if not ref or not pred:
        return 0.0
    lcs = lcs_len(ref, pred)
    prec = lcs / len(pred)
    rec = lcs / len(ref)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def exact_match(reference: str, prediction: str) -> int:
    return int(str(reference).strip().lower() == str(prediction).strip().lower())


In [ ]:
rows = []
visual_samples = []

with torch.inference_mode():
    for query_index, query_row in tqdm(test_df.iterrows(), total=len(test_df), desc='Frozen few-shot inference'):
        demos = select_demonstrations(query_row, query_index)
        messages = build_messages(query_row, demos)
        inputs = prepare_inputs(messages)
        generated = model.generate(
            **inputs,
            max_new_tokens=CFG.MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
        trimmed = generated[:, inputs['input_ids'].shape[1]:]
        prediction = processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0].strip().split('\n')[0].strip()

        target = query_row['answer']
        row = {
            'category': query_row['category'],
            'question': query_row['question'],
            'target': target,
            'prediction': prediction,
            'exact_match': exact_match(target, prediction),
            'bleu': sentence_bleu(target, prediction),
            'rouge_l': rouge_l(target, prediction),
            'num_shots': CFG.NUM_SHOTS,
            'support_image_paths': json.dumps([d['rgb_resolved'] for d in demos]),
        }
        rows.append(row)
        if len(visual_samples) < 6:
            visual_samples.append((query_row, prediction, row['exact_match']))

results = pd.DataFrame(rows)
results.to_csv(CFG.OUTPUT_DIR / 'few_shot_test_predictions.csv', index=False)
summary = {
    'model': CFG.MODEL,
    'num_shots': CFG.NUM_SHOTS,
    'weight_updates': 0,
    'n_samples': len(results),
    'exact_match': float(results['exact_match'].mean()),
    'bleu': float(results['bleu'].mean()),
    'rouge_l': float(results['rouge_l'].mean()),
}
with open(CFG.OUTPUT_DIR / 'few_shot_test_metrics.json', 'w') as handle:
    json.dump(summary, handle, indent=2)
print(json.dumps(summary, indent=2))

for index, (sample, prediction, em) in enumerate(visual_samples, 1):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
    axes[0].imshow(load_image(sample['rgb_resolved']))
    axes[0].set_title('RGB')
    axes[1].imshow(load_image(sample['thermal_resolved']))
    axes[1].set_title('Thermal')
    for axis in axes:
        axis.axis('off')
    plt.suptitle(
        f'{index}. {sample["question"]}\nTarget: {sample["answer"]} | Prediction: {prediction} | EM: {em}',
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()


In [ ]:
!zip -r /kaggle/working/firevlm_frozen_few_shot_qwen25vl_3b.zip /kaggle/working/firevlm_frozen_few_shot_qwen25vl_3b
